In [10]:
while True:

    print("Os conectivos disponíveis para uso são '∧', '∨', '→', '↔' e '¬'\nVocê pode copiá-los da frase acima")
    proposicao = input("Insira a proposição a ser analisada:")

    separados = []
    letras = set()

    for i in proposicao:
        if i in ['∧', '∨', '→', '↔', '(', ')', '¬']:
            separados.append(i)
        elif i.isalpha():
            separados.append(i)
            letras.add(i)

    letras = sorted(list(letras))

    def gerar_combinacoes(letras, atual=[], resultado=[]):
        if len(atual) == len(letras):
            resultado.append(atual.copy())
            return resultado

        gerar_combinacoes(letras, atual + [True], resultado)
        gerar_combinacoes(letras, atual + [False], resultado)

        return resultado

    combinacoes = gerar_combinacoes(letras)

    dados = {}

    for letra in letras:
        dados[letra] = []
        
    for linha in combinacoes:
        for i in range(len(letras)):
            dados[letras[i]].append(linha[i])

    def converte_rpn(separados):
        saida = []
        pilha = []

        precedencia = {
            '¬': 3,
            '∧': 2,
            '∨': 1,
            '→': 0,
            '↔': 0
        }

        for caractere in separados:

            if caractere.isalpha():
                saida.append(caractere)

            elif caractere == '(':
                pilha.append(caractere)

            elif caractere == ')':
                while pilha and pilha[-1] != '(':
                    saida.append(pilha.pop())
                pilha.pop()

            else:
                while ((pilha and pilha[-1] != '(') and (precedencia.get(pilha[-1], 0) >= precedencia.get(caractere, 0))):
                    saida.append(pilha.pop())

                pilha.append(caractere)

        while pilha:
            saida.append(pilha.pop())

        return saida

    rpn = converte_rpn(separados)

    def conjuncao(v1,v2):
        resultado = (v1 and v2)
        return resultado

    def disjuncao(v1,v2):
        resultado = (v1 or v2)
        return resultado

    def negacao(v):
        resultado = (not v)
        return resultado

    def condicional(v1,v2):
        resultado = (not(v1) or v2)
        return resultado

    def bicondicional(v1,v2):
        resultado = (condicional(v1,v2) and condicional(v2,v1))
        return resultado

    def avalia_rpn(rpn, valores):
        pilha_valores = []
        pilha_expressoes = []
        resultados = {}

        for caractere in rpn:

            if caractere.isalpha():
                pilha_valores.append(valores[caractere])
                pilha_expressoes.append(caractere)

            elif caractere == '¬':
                v = pilha_valores.pop()
                expr = pilha_expressoes.pop()

                pilha_valores.append(negacao(v))
                pilha_expressoes.append(f"¬{expr}")
                resultados[f"¬{expr}"] = negacao(v)


            else:
                v2 = pilha_valores.pop()
                v1 = pilha_valores.pop()

                e2 = pilha_expressoes.pop()
                e1 = pilha_expressoes.pop()

                if caractere == '∧':
                    resultado = conjuncao(v1,v2)

                elif caractere == '∨':
                    resultado = disjuncao(v1,v2)

                elif caractere == '→':
                    resultado = condicional(v1,v2)
                    pilha_expressoes.append(f"({e1}{caractere}{e2})")

                elif caractere == '↔':
                    resultado = bicondicional(v1,v2)
                        
                    

                pilha_valores.append(resultado)
                pilha_expressoes.append(f"({e1}{caractere}{e2})")
                resultados[f"({e1}{caractere}{e2})"] = resultado

        return resultados

    subexpressoes_colunas = {}

    for linha in combinacoes:
        valores = dict(zip(letras, linha))
        subexpressoes = avalia_rpn(rpn, valores)

        for expressao in subexpressoes: 
            if expressao not in subexpressoes_colunas:
                subexpressoes_colunas[expressao] = []
            subexpressoes_colunas[expressao].append(subexpressoes[expressao])

    for expressao in subexpressoes_colunas:
        dados[expressao] = subexpressoes_colunas[expressao]

    ordem = letras + list(subexpressoes_colunas.keys()) 

    import pandas as pd

    tabela = pd.DataFrame(dados)[ordem]
    print(tabela) 

    while True:
        quer_verificar = input("Gostaria de verificar se é uma tautologia, contradição ou nenhum dos casos? (s/n)")

        if quer_verificar.lower() == "s":
            if tabela.iloc[:, -1].all():
                print(f"{proposicao} é uma tautologia")
            elif not tabela.iloc[:, -1].any():
                print(f"{proposicao} é uma contradição")
            else:
                print(f"{proposicao} não é uma tautologia nem uma contradição")
            break

        elif quer_verificar.lower() == "n":
            break

        else:
            print("Não entendi sua resposta, tente novamente")

    quer_repetir = input("Gostaria de analisar uma outra porposição? (s/n)")
    if quer_repetir.lower() == "s":
            pass

    elif quer_repetir.lower() == "n":
        print("Programa encerrado. Quando quiser reiniciar, rode o código novamente")
        break

    else:
        print("Não entendi sua resposta, tente novamente")

Os conectivos disponíveis para uso são '∧', '∨', '→', '↔' e '¬'
Você pode copiá-los da frase acima
       a      b
0   True   True
1   True  False
2  False   True
3  False  False
Programa encerrado. Quando quiser reiniciar, rode o código novamente
